# Notebook 05: Preparacao para BI e Looker Studio

**Objetivo:** transformar a base tratada em um CSV pronto para o Looker Studio, seguindo o modelo estrela aprovado pela equipe em 05/11/2026.

**Entradas:**
- `data/processed/datatran_sul_tratado.csv`

**Saidas:**
- `data/looker/fato_acidentes.csv`: tabela fato com medidas e chaves
- `data/looker/dim_tempo.csv`
- `data/looker/dim_local.csv`
- `data/looker/dim_via.csv`
- `data/looker/dim_condicao.csv`
- `data/looker/dim_acidente.csv`

**Perguntas que este notebook responde:**
1. Os dados estao no formato correto para o Looker Studio?
2. Os 4 indicadores aprovados pela equipe podem ser calculados a partir da fato?
3. Os filtros por UF, BR e periodo funcionam via dimensoes?

**Modelo estrela aprovado (05/11/2026):**
- Granularidade: uma linha = um acidente em um momento
- Frase do evento: "Em [data], um acidente na BR [numero], km [km], em [municipio/UF], envolvendo [n] veiculos, resultou em [n] mortos e [n] feridos."

**Indicadores aprovados:**
1. (PRINCIPAL) Taxa de letalidade: mortos / total de acidentes x 100
2. Taxa de acidentes graves ou fatais: grave_ou_fatal / total x 100
3. Proporcao em condicao adversa: acidentes com chuva ou neblina / total x 100
4. Proporcao de acidentes noturnos: acidentes em Noite+Madrugada / total x 100

## 0. Imports e configuracao

In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Caminhos
ROOT    = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC    = os.path.join(ROOT, 'data', 'processed')
LOOKER  = os.path.join(ROOT, 'data', 'looker')
SRC     = os.path.join(ROOT, 'src')
sys.path.insert(0, SRC)

os.makedirs(LOOKER, exist_ok=True)

print('ROOT:', ROOT)
print('Pasta looker:', LOOKER)

ROOT: /home/gabrielr/AbexVI
Pasta looker: /home/gabrielr/AbexVI/data/looker


## 1. Carregamento da base tratada

Carregamos a base que saiu do NB02 (datatran_sul_tratado.csv). Esta e a unica fonte de dados deste notebook.

In [2]:
df = pd.read_csv(os.path.join(PROC, 'datatran_sul_tratado.csv'), low_memory=False)
print(f'Base carregada: {df.shape[0]} linhas x {df.shape[1]} colunas')
print('\nColunas disponíveis:')
print(list(df.columns))

Base carregada: 8703 linhas x 51 colunas

Colunas disponíveis:
['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'feridos', 'ilesos', 'ignorados', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop', 'mes', 'dia_semana_num', 'fim_de_semana', 'hora', 'faixa_horaria', 'tracado_via_norm', 'tracado_aclive', 'tracado_curva', 'tracado_declive', 'tracado_desvio_temporário', 'tracado_em_obras', 'tracado_intersecao_de_vias', 'tracado_ponte', 'tracado_reta', 'tracado_retorno_regulamentado', 'tracado_rotatoria', 'tracado_túnel', 'tracado_viaduto', 'macrocausa', 'teve_morte', 'grave_ou_fatal']


## 2. Modelo estrela: visao geral

O modelo estrela organiza os dados em:
- **Tabela fato** (centro): os eventos medidos. Uma linha = um acidente.
- **Tabelas dimensao** (pontas): o contexto de cada evento (quando, onde, como).

No Looker Studio, as dimensoes viram filtros e eixos dos graficos. As medidas da fato viram os numeros exibidos.

```
                dim_tempo
                    |
dim_acidente -- fato_acidentes -- dim_local
                    |
              dim_via / dim_condicao
```

## 3. Construcao das dimensoes

Cada dimensao recebe um identificador unico (id_) que serve de chave para ligar com a tabela fato.

### 3.1 dim_tempo

Captura o contexto temporal de cada acidente: data completa, mes, dia da semana, hora, faixa horaria e se foi fim de semana.

In [3]:
colunas_tempo = ['data_inversa', 'mes', 'dia_semana', 'dia_semana_num',
                 'hora', 'faixa_horaria', 'fim_de_semana']

dim_tempo = df[colunas_tempo].drop_duplicates().reset_index(drop=True)
dim_tempo.insert(0, 'id_tempo', range(1, len(dim_tempo) + 1))

print(f'dim_tempo: {dim_tempo.shape[0]} registros unicos')
dim_tempo.head()

dim_tempo: 3051 registros unicos


,id_tempo,data_inversa,mes,dia_semana,dia_semana_num,hora,faixa_horaria,fim_de_semana
0,1,2026-01-01,1,quinta-feira,3,6,Manha (6-11h),0
1,2,2026-01-01,1,quinta-feira,3,14,Tarde (12-17h),0
2,3,2026-01-01,1,quinta-feira,3,18,Noite (18-23h),0
3,4,2026-01-02,1,sexta-feira,4,5,Madrugada (0-5h),0
4,5,2026-01-02,1,sexta-feira,4,7,Manha (6-11h),0


### 3.2 dim_local

Captura onde o acidente ocorreu: estado, municipio, rodovia, quilometro, uso do solo e coordenadas geograficas.

In [4]:
colunas_local = ['uf', 'municipio', 'br', 'km', 'uso_solo',
                 'latitude', 'longitude', 'regional', 'delegacia', 'uop']

dim_local = df[colunas_local].drop_duplicates().reset_index(drop=True)
dim_local.insert(0, 'id_local', range(1, len(dim_local) + 1))

print(f'dim_local: {dim_local.shape[0]} registros unicos')
dim_local.head()

dim_local: 8680 registros unicos


,id_local,uf,municipio,br,km,uso_solo,latitude,longitude,regional,delegacia,uop
0,1,SC,BIGUACU,101,193.0,Sim,-27.489352,-48.655658,SPRF-SC,DEL01/SC,UOP01-DEL01-SC
1,2,SC,ITAJAI,101,127.0,Não,-26.970154,-48.683412,SPRF-SC,DEL04/SC,UOP04-DEL04-SC
2,3,PR,CAMPINA GRANDE DO SUL,116,51.1,Não,-25.243394,-48.902142,SPRF-PR,DEL01/PR,UOP02-DEL01-PR
3,4,SC,ARAQUARI,101,56.0,Sim,-26.435983,-48.812596,SPRF-SC,DEL03/SC,UOP01-DEL03-SC
4,5,PR,MAUA DA SERRA,376,301.0,Não,-23.936839,-51.166871,SPRF-PR,DEL07/PR,UOP02-DEL07-PR


### 3.3 dim_via

Captura as caracteristicas fisicas da via no ponto do acidente: tipo de pista, tracado e sentido.

In [5]:
colunas_via = ['tipo_pista', 'tracado_via_norm', 'sentido_via']

dim_via = df[colunas_via].drop_duplicates().reset_index(drop=True)
dim_via.insert(0, 'id_via', range(1, len(dim_via) + 1))

print(f'dim_via: {dim_via.shape[0]} registros unicos')
dim_via.head()

dim_via: 393 registros unicos


,id_via,tipo_pista,tracado_via_norm,sentido_via
0,1,Dupla,Curva,Crescente
1,2,Dupla,Reta,Crescente
2,3,Dupla,Curva,Decrescente
3,4,Dupla,Reta,Decrescente
4,5,Simples,Interseção de Vias;Reta;Rotatória,Decrescente


### 3.4 dim_condicao

Captura as condicoes ambientais no momento do acidente: clima e fase do dia (luz disponivel).

In [6]:
colunas_condicao = ['condicao_metereologica', 'fase_dia']

dim_condicao = df[colunas_condicao].drop_duplicates().reset_index(drop=True)
dim_condicao.insert(0, 'id_condicao', range(1, len(dim_condicao) + 1))

print(f'dim_condicao: {dim_condicao.shape[0]} registros unicos')
dim_condicao.head()

dim_condicao: 30 registros unicos


,id_condicao,condicao_metereologica,fase_dia
0,1,Nublado,Amanhecer
1,2,Céu Claro,Pleno dia
2,3,Chuva,Anoitecer
3,4,Chuva,Pleno dia
4,5,Nublado,Pleno dia


### 3.5 dim_acidente

Captura o tipo e a causa do acidente, incluindo a macrocausa criada no NB02.

In [7]:
colunas_acidente = ['tipo_acidente', 'causa_acidente', 'macrocausa']

dim_acidente = df[colunas_acidente].drop_duplicates().reset_index(drop=True)
dim_acidente.insert(0, 'id_acidente', range(1, len(dim_acidente) + 1))

print(f'dim_acidente: {dim_acidente.shape[0]} registros unicos')
dim_acidente.head()

dim_acidente: 416 registros unicos


,id_acidente,tipo_acidente,causa_acidente,macrocausa
0,1,Colisão lateral mesmo sentido,Reação tardia ou ineficiente do condutor,Falha de atencao ou reacao
1,2,Colisão frontal,Transitar no Acostamento,Outras
2,3,Saída de leito carroçável,Velocidade Incompatível,Comportamento de risco
3,4,Colisão lateral mesmo sentido,Manobra de mudança de faixa,Falha de atencao ou reacao
4,5,Saída de leito carroçável,Reação tardia ou ineficiente do condutor,Falha de atencao ou reacao


## 4. Construcao da tabela fato

A tabela fato tem: o identificador do acidente, as chaves FK para cada dimensao e as medidas numericas.
Cada linha continua sendo um acidente (granularidade preservada).

In [8]:
# Fazer o join da base original com cada dimensao para obter os ids
df_fato = df.copy()

df_fato = df_fato.merge(dim_tempo[['id_tempo'] + colunas_tempo], on=colunas_tempo, how='left')
df_fato = df_fato.merge(dim_local[['id_local'] + colunas_local], on=colunas_local, how='left')
df_fato = df_fato.merge(dim_via[['id_via'] + colunas_via], on=colunas_via, how='left')
df_fato = df_fato.merge(dim_condicao[['id_condicao'] + colunas_condicao], on=colunas_condicao, how='left')
df_fato = df_fato.merge(dim_acidente[['id_acidente'] + colunas_acidente], on=colunas_acidente, how='left')

# Selecionar apenas as colunas da fato
colunas_fato = [
    'id',
    'id_tempo', 'id_local', 'id_via', 'id_condicao', 'id_acidente',
    'classificacao_acidente',
    'mortos', 'feridos_graves', 'feridos_leves', 'ilesos',
    'veiculos', 'pessoas',
    'teve_morte', 'grave_ou_fatal'
]

fato_acidentes = df_fato[colunas_fato].copy()

print(f'fato_acidentes: {fato_acidentes.shape[0]} linhas x {fato_acidentes.shape[1]} colunas')
print(f'\nChaves FK com NaN:')
for col in ['id_tempo', 'id_local', 'id_via', 'id_condicao', 'id_acidente']:
    print(f'  {col}: {fato_acidentes[col].isna().sum()} ausentes')

fato_acidentes.head()

fato_acidentes: 8703 linhas x 15 colunas

Chaves FK com NaN:
  id_tempo: 0 ausentes
  id_local: 0 ausentes
  id_via: 0 ausentes
  id_condicao: 0 ausentes
  id_acidente: 0 ausentes


,id,id_tempo,id_local,id_via,id_condicao,id_acidente,classificacao_acidente,mortos,feridos_graves,feridos_leves,ilesos,veiculos,pessoas,teve_morte,grave_ou_fatal
0,742943,1,1,1,1,1,Com Vítimas Feridas,0,0,1,1,4,3,0,0
1,743001,2,2,2,2,2,Com Vítimas Feridas,0,0,1,1,2,2,0,0
2,743077,3,3,3,3,3,Com Vítimas Feridas,0,1,0,0,2,2,0,1
3,743112,4,4,4,1,4,Com Vítimas Feridas,0,0,1,1,6,4,0,0
4,743127,5,5,1,2,5,Com Vítimas Feridas,0,0,1,0,2,2,0,0


## 5. Validacao dos indicadores aprovados

Antes de exportar, verificamos se os 4 indicadores aprovados pela equipe podem ser calculados a partir da tabela fato. Isso confirma que o modelo estrela esta correto.

In [9]:
total = len(fato_acidentes)

# Indicador 1 (PRINCIPAL): Taxa de letalidade
taxa_letalidade = fato_acidentes['mortos'].sum() / total * 100

# Indicador 2: Taxa de acidentes graves ou fatais
taxa_grave_fatal = fato_acidentes['grave_ou_fatal'].sum() / total * 100

# Indicador 3: Proporcao em condicao adversa
# Juntamos fato com dim_condicao para filtrar por tipo de clima
condicoes_adversas = ['Chuva', 'Garoa/Chuvisco', 'Nevoeiro/Neblina', 'Neve', 'Vento']
fato_com_condicao = fato_acidentes.merge(
    dim_condicao[['id_condicao', 'condicao_metereologica']],
    on='id_condicao', how='left'
)
taxa_adversa = (fato_com_condicao['condicao_metereologica'].isin(condicoes_adversas)).sum() / total * 100

# Indicador 4: Proporcao de acidentes noturnos
fato_com_tempo = fato_acidentes.merge(
    dim_tempo[['id_tempo', 'faixa_horaria']],
    on='id_tempo', how='left'
)
periodos_noturnos = ['Noite (18-23h)', 'Madrugada (0-5h)']
taxa_noturna = (fato_com_tempo['faixa_horaria'].isin(periodos_noturnos)).sum() / total * 100

print('=== INDICADORES APROVADOS (jan-mai 2026, SC+PR+RS) ===')
print(f'Total de acidentes: {total:,}')
print()
print(f'(1-PRINCIPAL) Taxa de letalidade:          {taxa_letalidade:.1f} mortos por 100 acidentes')
print(f'(2) Taxa de acidentes graves ou fatais:    {taxa_grave_fatal:.1f}%')
print(f'(3) Proporcao em condicao adversa:         {taxa_adversa:.1f}%')
print(f'(4) Proporcao de acidentes noturnos:       {taxa_noturna:.1f}%')

=== INDICADORES APROVADOS (jan-mai 2026, SC+PR+RS) ===
Total de acidentes: 8,703

(1-PRINCIPAL) Taxa de letalidade:          6.1 mortos por 100 acidentes
(2) Taxa de acidentes graves ou fatais:    24.6%
(3) Proporcao em condicao adversa:         16.3%
(4) Proporcao de acidentes noturnos:       40.6%


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> Leia os quatro números acima e responda:
>
> - A taxa de letalidade esta acima ou abaixo do esperado para rodovias federais?
> - A proporcao de acidentes em condicao adversa (chuva, neblina) e alta? O que isso sugere para a PRF?
> - Mais de que fração dos acidentes ocorre a noite ou de madrugada? Isso justifica fiscalizacao noturna?
>
> Dica: compare os valores por UF (SC, PR, RS) no proximo bloco para ver se ha diferenca regional.

### 5.1 Indicadores por UF

Comparamos os 4 indicadores entre os tres estados. Taxas sao usadas (nao contagens absolutas) para evitar distorcao pelo volume de acidentes por estado.

In [10]:
# Juntamos fato com todas as dimensoes necessarias
fato_full = fato_acidentes.merge(
    dim_local[['id_local', 'uf']], on='id_local', how='left'
).merge(
    dim_condicao[['id_condicao', 'condicao_metereologica']], on='id_condicao', how='left'
).merge(
    dim_tempo[['id_tempo', 'faixa_horaria']], on='id_tempo', how='left'
)

fato_full['condicao_adversa'] = fato_full['condicao_metereologica'].isin(condicoes_adversas).astype(int)
fato_full['noturno']          = fato_full['faixa_horaria'].isin(periodos_noturnos).astype(int)

resumo_uf = fato_full.groupby('uf').agg(
    total_acidentes   = ('id', 'count'),
    total_mortos      = ('mortos', 'sum'),
    grave_ou_fatal    = ('grave_ou_fatal', 'sum'),
    condicao_adversa  = ('condicao_adversa', 'sum'),
    noturno           = ('noturno', 'sum')
).assign(
    taxa_letalidade   = lambda x: x['total_mortos']   / x['total_acidentes'] * 100,
    taxa_grave_fatal  = lambda x: x['grave_ou_fatal']  / x['total_acidentes'] * 100,
    taxa_adversa      = lambda x: x['condicao_adversa']/ x['total_acidentes'] * 100,
    taxa_noturna      = lambda x: x['noturno']         / x['total_acidentes'] * 100
)[['total_acidentes','taxa_letalidade','taxa_grave_fatal','taxa_adversa','taxa_noturna']]

print('Indicadores por UF (jan-mai 2026):')
print(resumo_uf.round(1).to_string())

Indicadores por UF (jan-mai 2026):
    total_acidentes  taxa_letalidade  taxa_grave_fatal  taxa_adversa  taxa_noturna
uf                                                                                
PR             3254              6.9              24.5          18.3          41.9
RS             1911              7.3              23.7          11.9          40.6
SC             3538              4.7              25.1          16.8          39.5


> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> Compare os tres estados na tabela acima:
>
> - Qual UF tem a maior taxa de letalidade? Isso e esperado pelo volume de trafego na regiao?
> - Ha diferenca na proporcao de acidentes noturnos entre os estados?
> - Um estado com mais chuva (clima mais chuvoso) aparece com maior proporcao de condicao adversa?
>
> Lembrete: sem dados de volume de trafego, nao e possivel afirmar que uma BR ou UF e mais "perigosa", apenas que registra mais acidentes ou maior taxa de mortos por acidente registrado.

## 6. Exportacao para o Looker Studio

Exportamos a tabela fato e as 5 dimensoes como CSVs separados. Configuracoes de exportacao:
- Decimal com ponto (padrao internacional, o Looker nao aceita virgula)
- Datas no formato ISO (AAAA-MM-DD)
- Encoding UTF-8
- Sem indice do pandas

In [11]:
EXPORT_OPTS = dict(index=False, encoding='utf-8')

fato_acidentes.to_csv(os.path.join(LOOKER, 'fato_acidentes.csv'),  **EXPORT_OPTS)
dim_tempo.to_csv(    os.path.join(LOOKER, 'dim_tempo.csv'),        **EXPORT_OPTS)
dim_local.to_csv(    os.path.join(LOOKER, 'dim_local.csv'),        **EXPORT_OPTS)
dim_via.to_csv(      os.path.join(LOOKER, 'dim_via.csv'),          **EXPORT_OPTS)
dim_condicao.to_csv( os.path.join(LOOKER, 'dim_condicao.csv'),     **EXPORT_OPTS)
dim_acidente.to_csv( os.path.join(LOOKER, 'dim_acidente.csv'),     **EXPORT_OPTS)

print('Arquivos exportados para data/looker/:')
for f in sorted(os.listdir(LOOKER)):
    caminho = os.path.join(LOOKER, f)
    linhas  = sum(1 for _ in open(caminho, encoding='utf-8')) - 1
    print(f'  {f}: {linhas:,} linhas')

Arquivos exportados para data/looker/:


  dim_acidente.csv: 416 linhas
  dim_condicao.csv: 30 linhas
  dim_local.csv: 8,680 linhas
  dim_tempo.csv: 3,051 linhas
  dim_via.csv: 393 linhas
  fato_acidentes.csv: 8,703 linhas


## 7. KPIs sugeridos para o dashboard

Tabela de referencia com os 4 indicadores aprovados pela equipe. Use esta tabela para configurar os campos calculados no Looker Studio.

In [12]:
kpis = pd.DataFrame([
    {
        'prioridade': '1 - PRINCIPAL',
        'nome': 'Taxa de letalidade',
        'formula': 'SUM(mortos) / COUNT(id) * 100',
        'periodo': 'Mensal',
        'referencia': 'Periodo anterior; meta < 5 mortos por 100 acidentes',
        'quem_usa': 'PRF: decide onde intensificar fiscalizacao e blitze'
    },
    {
        'prioridade': '2',
        'nome': 'Taxa de acidentes graves ou fatais',
        'formula': 'SUM(grave_ou_fatal) / COUNT(id) * 100',
        'periodo': 'Mensal',
        'referencia': 'Periodo anterior por UF',
        'quem_usa': 'PRF e DNIT: decide intervencoes na via'
    },
    {
        'prioridade': '3',
        'nome': 'Proporcao em condicao adversa',
        'formula': 'COUNT(id onde clima adverso) / COUNT(id) * 100',
        'periodo': 'Mensal',
        'referencia': 'Periodo anterior',
        'quem_usa': 'PRF: decide reforco de patrulhamento em dias de chuva'
    },
    {
        'prioridade': '4',
        'nome': 'Proporcao de acidentes noturnos',
        'formula': 'COUNT(id onde faixa_horaria em Noite ou Madrugada) / COUNT(id) * 100',
        'periodo': 'Mensal',
        'referencia': 'Periodo anterior',
        'quem_usa': 'PRF: decide reforco da fiscalizacao noturna'
    }
])

print('Tabela de indicadores aprovados:')
print(kpis.to_string(index=False))

Tabela de indicadores aprovados:
   prioridade                               nome                                                              formula periodo                                          referencia                                              quem_usa
1 - PRINCIPAL                 Taxa de letalidade                                        SUM(mortos) / COUNT(id) * 100  Mensal Periodo anterior; meta < 5 mortos por 100 acidentes   PRF: decide onde intensificar fiscalizacao e blitze
            2 Taxa de acidentes graves ou fatais                                SUM(grave_ou_fatal) / COUNT(id) * 100  Mensal                             Periodo anterior por UF                PRF e DNIT: decide intervencoes na via
            3      Proporcao em condicao adversa                       COUNT(id onde clima adverso) / COUNT(id) * 100  Mensal                                    Periodo anterior PRF: decide reforco de patrulhamento em dias de chuva
            4    Proporcao de acidentes

## 8. Resumo do notebook

**O que foi produzido:**
- Modelo estrela com 1 tabela fato e 5 dimensoes
- 6 CSVs exportados para `data/looker/`
- Validacao dos 4 indicadores aprovados pela equipe
- Comparativo dos indicadores por UF (SC, PR, RS)

**Proximo passo:** a equipe conecta os CSVs no Looker Studio (via Google Sheets ou upload direto) e monta o dashboard com os 4 indicadores.

**Limitacoes a declarar no dashboard:**
- Dados cobrem apenas jan-mai/2026 (5 meses); sazonalidade pode estar presente
- Sem dados de volume de trafego: taxas refletem acidentes registrados, nao risco relativo
- Apenas rodovias federais: nao inclui estaduais nem urbanas

> ✍️ **INTERPRETAÇÃO DA EQUIPE:**
>
> Revise os valores dos 4 indicadores calculados na secao 5 e escreva aqui uma frase-resumo para cada um, como apareceria no dashboard:
> Exemplo: "Em jan-mai/2026, a taxa de letalidade na Regiao Sul foi de X mortos por 100 acidentes, [acima/abaixo] da meta de 5."
>
> Dica: use esses valores como referencia inicial (baseline) para o mes seguinte quando o dashboard estiver ativo.